# Parte 1: Exploración

**!ESTA CELDA NO VA EN LA VERSIÓN FINAL!**<br>
<!-- IGNORAR CELDA -->
Esta primer parte de la libreta corresponde al primer y segundo inciso del proyecto.
[Link indicaciones proyecto](https://ml-unison.github.io/proyecto/).

## ¿Qué se predice?

Se predice el **TVT** (*True Vertical Thickness*), medido en pies, para cada pie de la parte horizontal de un pozo. El TVT indica **en qué capa de roca está la barrena en cada momento**. Para saberlo se compara con un pozo vertical de referencia (*typewell*), que funciona como un mapa de las capas: cada profundidad de ese pozo corresponde a un valor de TVT.

Aunque el pozo avance casi en línea recta, las capas de roca no son planas. Si el pozo cruza hacia capas más altas, el TVT sube; si cruza hacia capas más bajas, baja; y si sigue dentro de la misma capa, se mantiene igual.

En los datos de entrenamiento, el TVT lo escribió a mano un geólogo. Para hacerlo comparó la radiactividad natural de la roca (rayos gamma, `GR`) medida en el pozo horizontal con la medida en el pozo de referencia, porque cada capa tiene un patrón propio. En cada pozo conocemos el TVT hasta cierto punto (la columna `TVT_input`). Después de ese punto la columna está vacía (`NaN`) y ahí tenemos que predecirlo.

En resumen, se trata de **continuar el trabajo del geólogo**: dado lo que ya se sabe del pozo, estimar el TVT de los siguientes pies usando la trayectoria del pozo (`MD`, `X`, `Y`, `Z`), sus rayos gamma y los del pozo de referencia.

**TODO** 
- Agregar imagenes de los slides 5,6 y/o 7 del pptx.

## ¿Con qué datos?

Los datos son mediciones de pozos petroleros horizontales. Cada pozo tiene un identificador único de 8 caracteres (por ejemplo, `015fe0d2`) y viene con dos archivos:

- **El pozo horizontal** (`{WELLNAME}__horizontal_well.csv`). Tiene un renglón por cada pie perforado, con:
  - `MD`: cuánto mide el pozo desde la superficie hasta ese punto.
  - `X`, `Y`, `Z`: la posición de ese punto bajo tierra (`Z` es la profundidad).
  - `GR`: los rayos gamma, es decir, la radiactividad natural de la roca en ese punto. Algunos valores faltan (`NaN`).
  - `TVT_input`: el TVT conocido hasta el punto de inicio de predicción; después está vacío.
  - `TVT`: la respuesta que hay que predecir (solo en entrenamiento).
- **El pozo de referencia** (`{WELLNAME}__typewell.csv`). Es un pozo vertical cercano que sirve como mapa de las capas. Tiene:
  - `TVT`: la profundidad en ese pozo.
  - `GR`: los rayos gamma a esa profundidad.
  - `Geology`: el nombre de la capa de roca (por ejemplo, `EGFDL` o `BUDA`).

Los pozos están separados en dos carpetas:

- **`train/`**: pozos con el TVT completo, para entrenar. Además traen dos cosas que **no** vienen en prueba: la profundidad de seis capas de roca a lo largo del pozo (`ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA`) y una imagen (`.png`) con el dibujo del pozo y sus capas.
- **`test/`**: unos 200 pozos en los que el TVT está oculto después del punto de inicio de predicción. Son los que Kaggle usa para calificar.

Todas las distancias están en pies.

## ¿Con qué métrica?

La competencia usa el **RMSE** (*root mean squared error*, raíz del error cuadrático medio):

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

Para cada pie que hay que predecir se calcula la diferencia entre el TVT que marcó el geólogo ($y_i$) y el que predijimos ($\hat{y}_i$). Esas diferencias se elevan al cuadrado, se promedian sobre **todos los puntos de todos los pozos de prueba** y al final se saca la raíz.

### ¿Por qué tiene sentido?

- **El TVT es un número continuo.** No se trata de adivinar una categoría, sino de acercarse lo más posible a un valor en pies. Una métrica de regresión como el RMSE es la opción natural.
- **Se lee en pies.** Como el RMSE queda en las mismas unidades que el TVT, es fácil de interpretar: un RMSE de 5 quiere decir que, a grandes rasgos, nos equivocamos unos 5 pies en la posición de la barrena dentro de las capas.
- **Castiga más los errores grandes.** Al elevar al cuadrado, equivocarse por 20 pies pesa mucho más que equivocarse varias veces por 2. Eso coincide con lo que importa en la práctica: una desviación pequeña casi no cambia nada, pero creer que la barrena está en una capa cuando en realidad está en otra puede llevar a perforar fuera de la zona buena y desperdiciar recursos.
- **Encaja con cómo crecen los errores en este problema.** La predicción avanza pie por pie a partir del último TVT conocido, así que un error al principio puede ir creciendo a lo largo del pozo. El RMSE penaliza fuerte esas predicciones que se van alejando poco a poco.

Hay que tomar en cuenta un detalle: como el promedio se hace sobre puntos y no sobre pozos, **los pozos con una zona de evaluación más larga pesan más** en la calificación. Un pozo mal predicho en un tramo largo puede subir mucho el RMSE total.

## Carga de datos

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from datos.descargar import descargar
from datos.cargar import cargar_datos_csv as cargar

RAIZ = descargar()

### Archivos cargados

Cada pozo tiene un identificador (`well_id`) y dos archivos CSV. Los de cada tipo se juntan en una sola tabla.

- **`train_hw` / `test_hw`** (`*__horizontal_well.csv`): registro del pozo horizontal por profundidad medida (`MD`), con coordenadas `X`, `Y`, `Z` y rayos gamma (`GR`). En entrenamiento también vienen las cimas de las formaciones (`ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA`) y el `TVT` real, que es la variable a predecir.
- **`train_tw` / `test_tw`** (`*__typewell.csv`): pozo tipo (de referencia) con `GR` por `TVT` y, en entrenamiento, la columna `Geology`.
- **`sample_sub`** (`sample_submission.csv`): formato de entrega; un `id` (`<pozo>_<fila>`) y el `tvt` predicho.

In [ ]:
train_hw = cargar(RAIZ, 'train', 'horizontal_well')
train_tw = cargar(RAIZ, 'train', 'typewell')
test_hw = cargar(RAIZ, 'test', 'horizontal_well')
test_tw = cargar(RAIZ, 'test', 'typewell')
sample_sub = pd.read_csv(RAIZ / 'sample_submission.csv')

In [3]:
for nombre, df in [('train_hw', train_hw), ('train_tw', train_tw), ('test_hw', test_hw), ('test_tw', test_tw), ('sample_sub', sample_sub)]:
    print(nombre, df.shape)
print('pozos train:', train_hw.well_id.nunique())
train_hw.head()

train_hw (5092255, 14)
train_tw (1567045, 4)
test_hw (19221, 7)
test_tw (5798, 3)
sample_sub (14151, 2)
pozos train: 773


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input,well_id
0,11467.0,2983525.16,1069022.09,-9258.57,-9395.81,-9569.86,-9597.64,-9670.99,-9705.96,-9846.35,11236.02,115.692586,11236.02,000d7d20
1,11468.0,2983525.18,1069022.30,-9259.55,-9395.75,-9569.80,-9597.58,-9670.93,-9705.90,-9846.29,11237.05,115.584293,11237.05,000d7d20
2,11469.0,2983525.20,1069022.52,-9260.52,-9395.69,-9569.74,-9597.52,-9670.87,-9705.84,-9846.23,11238.09,135.446960,11238.09,000d7d20
3,11470.0,2983525.22,1069022.73,-9261.50,-9395.64,-9569.69,-9597.47,-9670.82,-9705.79,-9846.18,11239.12,140.401346,11239.12,000d7d20
4,11471.0,2983525.25,1069022.95,-9262.47,-9395.58,-9569.63,-9597.41,-9670.76,-9705.73,-9846.12,11240.15,111.270638,11240.15,000d7d20


### Columnas y valores faltantes: train vs test

Se compara qué columnas de `train_hw` existen en `test_hw` y qué porcentaje de cada una está vacío.

- **Solo en train:** `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA` y `TVT` no vienen en test. Las cimas de las formaciones no se pueden usar como variables al predecir, y `TVT` es el objetivo.
- **Disponibles en test:** `MD`, `X`, `Y`, `Z` y `well_id` están completas (0 % faltante), así que la geometría del pozo es información confiable.
- **`GR`:** tiene faltantes en ambos conjuntos (29.6 % en train, 19.7 % en test), por lo que habrá que decidir cómo tratarlo.
- **`TVT_input`:** falta en ~74 % de las filas, tanto en train (74.3 %) como en test (73.6 %). Solo se conoce el `TVT` en ~26 % de las filas de cada pozo. El porcentaje faltante en test (73.6 %) coincide con la proporción de filas de `sample_sub` respecto a `test_hw` (14151 de 19221), que es lo que vamos a predecir: el resto del pozo a partir del tramo con `TVT_input` conocido.

In [4]:
faltantes = pd.DataFrame({
    'train_%': train_hw.isna().mean() * 100,
    'test_%': test_hw.isna().mean() * 100,
}).round(2)
faltantes['en_test'] = faltantes.index.isin(test_hw.columns)
faltantes.sort_values(['en_test', 'test_%'], ascending=[False, False])

,train_%,test_%,en_test
TVT_input,74.31,73.62,True
GR,29.61,19.69,True
MD,0.00,0.00,True
X,0.00,0.00,True
Y,0.00,0.00,True
Z,0.00,0.00,True
well_id,0.00,0.00,True
ANCC,0.90,NaN,False
ASTNL,0.00,NaN,False
ASTNU,0.00,NaN,False
